# HiQuE: 질문별 멀티모달 우울증 예측

인터뷰의 **질문별 음성(A)·얼굴 움직임(V)·답변 텍스트(T)**를 모아 참가자의 PHQ-8 이진 선별 라벨을 예측한다. 이 노트북에서는 데이터가 모델 입력으로 바뀌고, 학습과 평가로 이어지는 과정을 단계별로 살펴본다.

```text
인터뷰 → 질문별 답변 구간 → 음성·얼굴·텍스트 특징 → 질문 간 attention
       → 모달리티 간 attention → 클래스 확률 → 평가
```

각 단계는 **기초 설명 → 논문·GitHub 코드 연결 → 실행 셀** 순서로 구성된다. 실행 코드는 저장소의 `src/urop`를 사용한다.

**사용 방법**

- 기본 `RUN_PIPELINE=False`: 저장된 입력·모델 구조·결과를 확인한다.
- 새 학습: 새 `RUN_NAME`과 `RUN_PIPELINE=True`를 설정한다.
- 학습 후 test 평가: `RUN_TEST=True`를 설정한다.

저장소 또는 이 실험 폴더에서 열고 위에서부터 실행한다. 새 학습 전에는 노트북을 저장한다.

## 0. 논문과 소스 안내

논문은 [HiQuE, arXiv:2408.03648v1](https://arxiv.org/html/2408.03648v1)이며, GitHub 링크는 [커밋 `24c553b`](https://github.com/JuHo-Jung/HiQuE/tree/24c553bf2666b442ae5b0e3490b998a5d4493559)를 기준으로 한다.

| 파이프라인 | 논문 | GitHub 소스 |
|---|---|---|
| 데이터·분할 | §3 | [`code/hique.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/hique.py) |
| 질문·응답 정렬 | §4.3, 부록 A | [`code/preprocess.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/preprocess.py), [`code/similarity.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/similarity.py) |
| 음성 특징 | §4.4.1 | [`code/feature_extraction/extract_audio.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/feature_extraction/extract_audio.py) |
| 얼굴 특징 | §4.4.2 | [`code/feature_extraction/extract_visual.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/feature_extraction/extract_visual.py) |
| 텍스트 특징 | §4.4.3 | [`embedding/roberta.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/embedding/roberta.py) |
| 질문 표현·모달리티 융합·분류 | §4.5–4.7 | [`code/fusionmodel.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/fusionmodel.py) |
| 증강·학습·평가 | §3.1, §5 | [`code/hique.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/hique.py)의 학습·평가 흐름 |

이 노트북의 E0–E4는 **같은 모델에 넣는 답변 입력을 달리하는 다섯 조건**이다. 모델은 고정 질문 위치를 사용하는 F_AVT이며, 논문의 계층 위치 표현과의 연결은 모델 단계에서 설명한다. 각 단계의 실행 구현 경로는 `src/urop/` 기준이다.

## 1. 실행 환경

**기초 설명**

노트북은 각 단계를 순서대로 호출한다. 음성·얼굴·텍스트 특징은 특징 추출용 Python 환경에서, 분류 모델의 학습은 TensorFlow 환경에서 실행한다. RoBERTa는 로컬에 저장된 가중치를 읽는다.

| 역할 | 환경 |
|---|---|
| 특징 추출 | `.tmp/hique-features/bin/python` |
| 모델 학습·평가 | `.tmp/hique-tf/bin/python` |
| 결과 저장 | `Data/runs/<RUN_NAME>/` |

`TEXT_DEVICE="auto"`는 CUDA에 접근할 수 있으면 GPU, 그렇지 않으면 CPU를 선택한다. 아래 셀은 실행 경로와 라이브러리·장치를 확인한다.

In [1]:
from pathlib import Path
from collections import Counter
import csv
import hashlib
import json
import os
import re
import shlex
import subprocess
import sys
from IPython.display import Markdown, display

RUN_PIPELINE = False
RUN_TEST = False
RUN_NAME = "hique_notebook_20261004"  # 새 설계/새 실행에는 새 이름 사용
TEXT_DEVICE = "auto"  # "auto", "cuda", "cpu"
AV_WORKERS = 4

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/urop/__main__.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("저장소 안에서 노트북을 열어주세요.")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
from urop.io import digest, require_private_output
from urop.paths import ROOT as PACKAGE_ROOT
assert ROOT == PACKAGE_ROOT
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", RUN_NAME)
if RUN_TEST and not RUN_PIPELINE:
    raise ValueError("RUN_TEST는 새 파이프라인 실행(RUN_PIPELINE=True)에서 사용합니다.")

HISTORY = ROOT / "Data/hique_input_study_v3"
BASE = require_private_output(ROOT / "Data/runs" / RUN_NAME)
VIEW = BASE if RUN_PIPELINE else HISTORY
ZIP_DIR = ROOT / "Data/DAIC-WOZ"
BASELINE = ROOT / "Data/hique_reproduction_excluding440/features.npz"
BASELINE_SEGMENTS = ROOT / "Data/hique_reproduction_excluding440/segments.jsonl"
QUESTIONS = ROOT / "Data/hique_reproduction/questions.json"
MODEL_DIR = ROOT / ".tmp/hique_weights/roberta-base"
FEATURE_PY = ROOT / ".tmp/hique-features/bin/python"
TRAIN_PY = ROOT / ".tmp/hique-tf/bin/python"
NOTEBOOK = ROOT / "experiments/hique/v3_input_study/pipeline.ipynb"
CONDITIONS = ["E0", "E1", "E2", "E3", "E4"]
SEEDS = [42, 13, 23, 37, 79]
assert BASE != HISTORY and not BASE.is_relative_to(HISTORY)
print("모드:", "새 실행" if RUN_PIPELINE else "기존 결과 읽기")
print("읽기 경로:", VIEW.relative_to(ROOT))
print("새 실행 출력:", BASE.relative_to(ROOT))

모드: 기존 결과 읽기
읽기 경로: Data/hique_input_study_v3
새 실행 출력: Data/runs/hique_notebook_20261004


In [2]:
ENV = {**os.environ, "PYTHONPATH": str(ROOT / "src"), "PYTHONDONTWRITEBYTECODE": "1",
       "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
       "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "2",
       "TF_CPP_MIN_LOG_LEVEL": "2"}

def read_json(path):
    return json.loads(Path(path).read_text())

def write_once(path, payload):
    path = require_private_output(Path(path))
    if path.exists():
        if read_json(path) != payload:
            raise ValueError(f"기존 설정이 다릅니다. 새 RUN_NAME을 사용하세요: {path}")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False) + "\n")

def python_read(python, source, *args):
    result = subprocess.run([str(python), "-c", source, *map(str, args)], cwd=ROOT,
                            env=ENV, text=True, capture_output=True, check=True)
    return result.stdout.strip()

def run_cli(python, command, *args, enabled=None):
    enabled = RUN_PIPELINE if enabled is None else enabled
    cmd = [str(python), "-m", "urop", command, *map(str, args)]
    print(shlex.join(cmd))
    if not enabled:
        print("[미실행] 설정을 바꾸면 이 명령이 실행됩니다.")
        return
    logs = BASE / "notebook_logs"
    logs.mkdir(parents=True, exist_ok=True)
    stage = str(args[args.index("--stage") + 1]) if "--stage" in args else "report"
    log = logs / f"{command}_{stage}.log"
    print("실행 중. 상세 로그:", log.relative_to(ROOT), flush=True)
    with log.open("a") as stream:
        result = subprocess.run(cmd, cwd=ROOT, env=ENV, stdout=stream, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f"종료 코드 {result.returncode}: {log}를 확인하세요.")
    print("완료:", command, stage)

def table(headers, rows):
    display(Markdown("| " + " | ".join(headers) + " |\n|" + "---|" * len(headers)
                     + "\n" + "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)))

def show_counts(path, keys):
    record = read_json(path)
    print(json.dumps({key: record[key] for key in keys if key in record}, ensure_ascii=False, indent=2))

In [3]:
required = [FEATURE_PY, TRAIN_PY, BASELINE, BASELINE_SEGMENTS, QUESTIONS,
            HISTORY / "mapping.json", HISTORY / "mapping_review.json", MODEL_DIR / "tokenizer.json"]
missing = [str(p.relative_to(ROOT)) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("필수 로컬 자료가 없습니다: " + ", ".join(missing))
from urop.models.upstream import SOURCE, SOURCE_SHA256, COMMIT
assert digest(SOURCE) == SOURCE_SHA256, "공개 모델 소스가 고정 버전과 다릅니다."
print("공개 코드 commit:", COMMIT)
feature_env = json.loads(python_read(FEATURE_PY, """
import json, sys, torch, transformers, opensmile
print(json.dumps({'python':sys.version.split()[0], 'torch':torch.__version__,
 'transformers':transformers.__version__, 'opensmile':opensmile.__version__,
 'cuda':torch.cuda.is_available(),
 'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))
"""))
print("특징 환경:", feature_env)
print("학습 환경:", python_read(TRAIN_PY, """
import json, sys
from urop.models.runtime import configure_tf
tf=configure_tf()
print(json.dumps({'python':sys.version.split()[0], 'tensorflow':tf.__version__,
 'gpu':[d.name for d in tf.config.list_physical_devices('GPU')]}))
"""))
if TEXT_DEVICE == "auto":
    DEVICE = "cuda" if feature_env["cuda"] else "cpu"
elif TEXT_DEVICE in ("cuda", "cpu"):
    DEVICE = TEXT_DEVICE
else:
    raise ValueError("TEXT_DEVICE는 auto/cuda/cpu 중 하나여야 합니다.")
if RUN_PIPELINE and DEVICE == "cuda" and not feature_env["cuda"]:
    raise RuntimeError("현재 프로세스에서 CUDA에 접근할 수 없습니다.")
print("선택된 텍스트 특징 장치:", DEVICE)

공개 코드 commit: 24c553bf2666b442ae5b0e3490b998a5d4493559


특징 환경: {'python': '3.13.5', 'torch': '2.10.0+cu128', 'transformers': '5.9.0', 'opensmile': '2.6.0', 'cuda': True, 'gpu': 'NVIDIA GeForce RTX 4070 Ti SUPER'}


학습 환경: {"python": "3.10.20", "tensorflow": "2.15.1", "gpu": []}
선택된 텍스트 특징 장치: cuda


## 2. 데이터 분할과 라벨

**기초 설명**

Train은 모델의 가중치를 학습하고, dev는 학습 중 모델을 선택하며, test는 선택된 모델을 평가한다. 한 사람의 답변이 여러 분할에 섞이지 않도록 **참가자 단위**로 나눈다. 예측 대상은 공식 CSV의 `PHQ8_Binary`다.

| 분할 | 논문 인원 | 노트북 인원 | 용도 |
|---|---:|---:|---|
| Train | 107 | 107 | 가중치 학습 |
| Dev | 35 | 32 | 최저 dev loss 모델 선택 |
| Test | 47 | 45 | 고정 모델 평가 |

노트북에서는 dev의 440(손상 ZIP), 451·458(질문 전사 부재), test의 411(E0 매핑 답변 부재), 480(질문 전사 부재)을 제외한다. 모든 입력 조건은 동일한 참가자를 사용한다.

**논문·코드 연결**

- 논문: §3: DAIC-WOZ와 공식 분할
- GitHub: [`code/hique.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/hique.py)의 split CSV 로딩
- 실행 구현: `data/transcripts.py:cohort`, `training/runner.py:load_roster`

In [4]:
from urop.data.transcripts import cohort
roster = cohort(ZIP_DIR)
assert Counter(roster.values()) == {"train": 107, "dev": 32}
assert not ({440, 451, 458} & set(roster))
missing_zips = [pid for pid in roster if not (ZIP_DIR / f"{pid}_P.zip").is_file()]
assert not missing_zips, f"필요한 ZIP 누락: {missing_zips}"
counts = []
for split in ("train", "dev"):
    with (ZIP_DIR / f"{split}_split_Depression_AVEC2017.csv").open(newline="") as stream:
        rows = [r for r in csv.DictReader(stream) if r.get("Participant_ID", "").strip()
                and int(r["Participant_ID"]) in roster]
    labels = Counter(int(r["PHQ8_Binary"]) for r in rows)
    assert set(labels) <= {0, 1}
    counts.append([split, len(rows), labels[0], labels[1]])
table(["분할", "인원", "음성 라벨 0", "양성 라벨 1"], counts)

| 분할 | 인원 | 음성 라벨 0 | 양성 라벨 1 |
|---|---|---|---|
| train | 107 | 77 | 30 |
| dev | 32 | 21 | 11 |

## 3. 질문 사전과 입력 조건

**기초 설명**

서로 다른 인터뷰를 비교하려면 같은 질문의 답변을 같은 위치에 놓아야 한다. 이를 위해 **주질문 66개와 후속질문 19개를 합친 85개 슬롯**을 사용한다. 슬롯은 질문 하나에 배정된 배열의 한 칸이다.

답변을 얻는 방법과 반복 답변의 선택 방식에 따라 입력을 나눈다.

| 조건 | 답변 출처·선택 | 텍스트 처리 |
|---|---|---|
| E0 | 음성 인식(ASR) 전사·마지막 답변 | 처음 510 내용 토큰 + 특수 토큰 2개 |
| E1 | 제공 전사·마지막 답변 | E0와 동일 |
| E2 | 제공 전사·첫 답변 | E0와 동일 |
| E3 | 제공 전사·모든 답변을 시간순 연결 | E0와 동일 |
| E4 | E3와 같은 답변 | 510토큰씩 나누어 처리 후 길이 가중 평균 |

E1−E0는 전사·화자·시간·매핑을 포함한 입력 경로, E2−E1은 첫/마지막 답변, E3−E1은 답변 수집 범위, E4−E3은 장문 처리의 비교다. 질문 매핑은 train 질문에 기반한 AI 검토 사전을 고정해 사용한다.

**논문·코드 연결**

- 논문: §4.3·부록 A: 85개 질문; §5.4: 새로운 질문의 매핑
- GitHub: [`code/preprocess.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/preprocess.py), [`code/similarity.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/similarity.py), [`code/whisper_segment.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/whisper_segment.py)
- 실행 구현: `data/transcripts.py:inventory/build_record`

E0는 저장된 ASR 결과를 읽는다. E1–E4의 응답 선택·장문 처리는 이 노트북의 입력 비교 조건이다.

In [5]:
protocol = {
    "scope": "V3 input study rerun from current src/urop; not exact paper reproduction",
    "conditions": CONDITIONS, "seeds": SEEDS, "cohort": {"train": 107, "dev": 32},
    "dev_exclusions": [440, 451, 458], "model": "F_AVT fixed positions 0..84",
    "epochs": 100, "batch_size": 8, "learning_rate": 0.0002, "dropout": 0.5,
    "normalization": "none", "checkpoint": "minimum dev loss", "threshold": "p1 > 0.5",
    "primary_comparison": "E1_minus_E0", "mapping": "frozen existing AI draft; no human gold",
    "baseline_policy": "reuse E0 ASR and A/V; recompute all T using corrected BPE",
    "upstream_commit": COMMIT,
}
if RUN_PIPELINE:
    write_once(BASE / "protocol.json", protocol)
    for name in ("mapping.json", "mapping_review.json"):
        destination = BASE / name
        if destination.exists() and digest(destination) != digest(HISTORY / name):
            raise ValueError(f"동결 매핑이 다릅니다: {destination}")
        if not destination.exists():
            destination.write_bytes((HISTORY / name).read_bytes())
    # 실행 출력/실행번호/설정 셀은 제외한다. RUN_TEST 전환이 train 레시피를 바꾸지 않는다.
    # 실험 의미를 결정하는 설정과 입력 hash는 아래 payload 및 protocol에 별도로 고정한다.
    saved = read_json(NOTEBOOK)
    code_sources = ["".join(c["source"]) for c in saved["cells"]
                    if c["cell_type"] == "code" and "settings" not in c.get("metadata", {}).get("tags", [])]
    write_once(BASE / "notebook_recipe.json", {
        "code_sha256": hashlib.sha256(json.dumps(code_sources).encode()).hexdigest(),
        "protocol": protocol, "text_device": DEVICE, "av_workers": AV_WORKERS,
        "baseline_sha256": digest(BASELINE), "baseline_segments_sha256": digest(BASELINE_SEGMENTS),
        "questions_sha256": digest(QUESTIONS), "mapping_sha256": digest(HISTORY / "mapping.json")})
run_cli(FEATURE_PY, "prepare", "--stage", "inventory", "--base", BASE, "--zip-dir", ZIP_DIR)
print("질문 코드북 항목 수:", len(read_json(QUESTIONS)))
print("고정 매핑 SHA256:", digest(VIEW / "mapping.json"))

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-features/bin/python -m urop prepare --stage inventory --base /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004 --zip-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ
[미실행] 설정을 바꾸면 이 명령이 실행됩니다.
질문 코드북 항목 수: 85
고정 매핑 SHA256: b70e1f06a848335a9287c9da8e3d059ec0dda90a56a80d65a3f187ba459fad78


## 4. 질문과 답변 구간 연결

**기초 설명**

전사에는 발화의 **화자, 시작·종료 시각, 문장**이 들어 있다. 면담자의 질문 뒤에 이어지는 참가자의 발화를 모으면, 질문별 답변 문장과 시간 구간을 얻을 수 있다.

`build_record`는 질문과 답변을 연결하고, `select_record`는 반복 답변 중 첫 답변·마지막 답변·전체 답변을 선택한다. 선택한 시간 구간은 음성·얼굴 특징 추출에도 사용한다. 따라서 세 모달리티가 같은 답변을 표현하게 된다.

| 입력 | 출력 |
|---|---|
| 화자·시각이 있는 전사와 질문 사전 | 질문 슬롯, 답변 텍스트, 답변 시간 구간 |

**논문·코드 연결**

- 논문: §4.3: 질문 중심의 인터뷰 구성
- GitHub: [`code/preprocess.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/preprocess.py)의 topic별 응답 구성
- 실행 구현: `data/transcripts.py:build_record/select_record`

In [6]:
run_cli(FEATURE_PY, "prepare", "--stage", "prepare", "--base", BASE,
        "--zip-dir", ZIP_DIR, "--mapping", BASE / "mapping.json", "--questions", QUESTIONS)
if RUN_PIPELINE:
    old = [json.loads(line) for line in BASELINE_SEGMENTS.read_text().splitlines() if line.strip()]
    selected = [r for r in old if int(r["participant_id"]) in roster]
    selected.sort(key=lambda r: int(r["participant_id"]))
    assert len(selected) == len({r["participant_id"] for r in selected}) == 139
    assert all(r["split"] == roster[int(r["participant_id"])] for r in selected)
    content = "".join(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n" for r in selected)
    path = BASE / "segments_E0.jsonl"
    if path.exists() and path.read_text() != content:
        raise ValueError("E0 기록이 변경됐습니다. 새 RUN_NAME을 사용하세요.")
    if not path.exists():
        path.write_text(content)
show_counts(VIEW / "preparation_summary.json", ["participants", "split_counts", "all_rows_accounted", "test_accessed"])
print(python_read(FEATURE_PY, """
from pathlib import Path
from urop.features.pipeline import read_records
import json, sys
records=read_records(Path(sys.argv[1]))
print(json.dumps({c:{'participants':len(rows),'question_slots':sum(len(r['slots']) for r in rows)}
                  for c,rows in records.items()}))
""", VIEW))

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-features/bin/python -m urop prepare --stage prepare --base /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004 --zip-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --mapping /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/mapping.json --questions /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction/questions.json
[미실행] 설정을 바꾸면 이 명령이 실행됩니다.
{
  "participants": 139,
  "split_counts": {
    "dev": 32,
    "train": 107
  },
  "all_rows_accounted": true,
  "test_accessed": false
}
{"E0": {"participants": 139, "question_slots": 3091}, "E1": {"participants": 139, "question_slots": 5322}, "E2": {"participants": 139, "question_slots": 5322}, "E3": {"participants": 139, "question_slots": 5322}}


## 5. 음성·얼굴 특징 추출

**기초 설명**

특징 벡터는 음성이나 얼굴 움직임을 모델이 계산할 수 있는 숫자로 요약한 것이다.

| 모달리티 | 추출 과정 | 참가자 한 명의 배열 |
|---|---|---|
| 음성 A | 답변 구간의 음성 → openSMILE eGeMAPSv02 → 음높이·에너지·음질 등의 88개 수치 | `85 × 88` |
| 얼굴 V | CLNF 얼굴 68개 점의 X/Y → 초당 유효 프레임 선택 → 좌표별 평균·분산 | `85 × 272` |

얼굴은 `68개 점 × 2개 좌표 × 2개 통계량 = 272차원`이다. 관측되지 않은 모달리티는 0 벡터와 관측 여부를 나타내는 mask로 표현한다. E4는 E3와 같은 음성·얼굴 특징을 사용한다.

**논문·코드 연결**

- 논문: §4.4.1: eGeMAPS 음성 특징; §4.4.2: CLNF 얼굴 특징
- GitHub: [`code/feature_extraction/extract_audio.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/feature_extraction/extract_audio.py), [`code/feature_extraction/extract_visual.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/feature_extraction/extract_visual.py)
- 실행 구현: `features/av.py:audio_samples/visual_features/extract_one`

공개 영상 추출 파일은 VGG 특징을 사용한다. 이 노트북의 얼굴 특징은 논문 §4.4.2의 CLNF 272차원 정의를 따른다.

In [7]:
feature_args = ["--base", BASE, "--baseline", BASELINE, "--model-dir", MODEL_DIR,
                "--zip-dir", ZIP_DIR, "--device", DEVICE, "--workers", AV_WORKERS]
run_cli(FEATURE_PY, "features", *feature_args, "--stage", "av")
print("A: (139, 85, 88), V: (139, 85, 272); E1/E2/E3 추출, E4는 E3 공유, E0는 기존 A/V")

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-features/bin/python -m urop features --base /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004 --baseline /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction_excluding440/features.npz --model-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique_weights/roberta-base --zip-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --device cuda --workers 4 --stage av
[미실행] 설정을 바꾸면 이 명령이 실행됩니다.
A: (139, 85, 88), V: (139, 85, 272); E1/E2/E3 추출, E4는 E3 공유, E0는 기존 A/V


## 6. 텍스트 특징 추출

**기초 설명**

Tokenizer는 답변 문장을 토큰 ID로 바꾼다. RoBERTa는 이 토큰들의 문맥을 계산한다. 마지막 층의 첫 토큰 위치(CLS)의 **768차원 벡터**를 답변의 표현으로 사용한다. 질문 85개의 답변을 모으면 `85 × 768` 배열이 된다.

RoBERTa 가중치는 고정하고, 이후 분류 모델만 학습한다. E0–E3은 내용 토큰 510개와 특수 토큰 2개를 입력한다. E4는 긴 답변을 510토큰 단위로 나누고, 각 블록의 CLS 벡터를 내용 토큰 수에 비례해 평균한다.

**논문·코드 연결**

- 논문: §4.4.3: RoBERTa의 마지막 층 CLS 표현
- GitHub: [`embedding/roberta.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/embedding/roberta.py)
- 실행 구현: `features/text.py:load_tokenizer/extract_text/token_blocks/pool_blocks`

공개 파일은 pooled 출력을 사용하며, 이 노트북은 논문의 마지막 층 CLS 정의를 따른다. E4의 블록 평균은 장문 처리 비교를 위한 조건이다.

In [8]:
# 합성 문장으로 tokenizer를 확인한다. 실제 참가자 발화는 표시하지 않는다.
print(python_read(FEATURE_PY, """
from pathlib import Path
from urop.features.text import load_tokenizer, token_blocks
import json, sys
tokenizer=load_tokenizer(Path(sys.argv[1]))
print(json.dumps({'tokenizer':'serialized BPE parity passed',
 'synthetic_ids':tokenizer('Hello world, this is a simple test.')['input_ids'],
 'example_1021_content_token_blocks':[len(x) for x in token_blocks(list(range(1021)))]}))
""", MODEL_DIR))
run_cli(FEATURE_PY, "features", *feature_args, "--stage", "text")
show_counts(VIEW / "features/text_audit.json", ["tokenizer_serialized_parity_verified",
    "E0_text_recomputed_correct_BPE", "short_E3_E4_bit_identical", "content_coverage_exact", "frozen_model"])

{"tokenizer": "serialized BPE parity passed", "synthetic_ids": [0, 31414, 232, 6, 42, 16, 10, 2007, 1296, 4, 2], "example_1021_content_token_blocks": [510, 510, 1]}
/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-features/bin/python -m urop features --base /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004 --baseline /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction_excluding440/features.npz --model-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique_weights/roberta-base --zip-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --device cuda --workers 4 --stage text
[미실행] 설정을 바꾸면 이 명령이 실행됩니다.
{
  "tokenizer_serialized_parity_verified": true,
  "E0_text_recomputed_correct_BPE": true,
  "short_E3_E4_bit_identical": true,
  "content_coverage_exact": true,
  "frozen_model": true
}


## 7. 모델 입력 배열 구성

**기초 설명**

질문별 특징을 참가자 순서대로 쌓으면 모델 입력이 된다. `N`은 참가자 수다. 모든 조건에서 참가자 순서와 질문 슬롯의 의미가 같아야 한다.

| 배열 | 크기 | 의미 |
|---|---|---|
| A | `N × 85 × 88` | 음성 특징 |
| V | `N × 85 × 272` | 얼굴 특징 |
| T | `N × 85 × 768` | 텍스트 특징 |
| mask_A / mask_V / mask_T | `N × 85` | 각 모달리티의 관측 여부 |
| question_presence | `N × 85` | 질문의 존재 여부 |
| position_ids | `N × 85` | 질문 위치 0–84 |

여기서는 특징을 표준화하지 않는다. mask는 결측 특징을 0으로 처리하는 데 사용하며, attention 자체를 차단하는 mask는 아니다. 아래 셀은 배열의 크기·값·참가자 정렬을 확인한다.

**논문·코드 연결**

- 논문: §4.1·4.4: 질문별 멀티모달 입력
- GitHub: [`code/preprocess.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/preprocess.py), [`code/fusionmodel.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/fusionmodel.py)의 입력 shape
- 실행 구현: `features/pipeline.py:assemble/validate_arrays`, `training/runner.py:load_conditions`

In [9]:
run_cli(FEATURE_PY, "features", *feature_args, "--stage", "assemble")
print(python_read(FEATURE_PY, """
from pathlib import Path
import json, sys, numpy as np
from urop.training.runner import load_roster, load_conditions
from urop.features.pipeline import validate_arrays
from urop.io import digest
base=Path(sys.argv[1]); roster,_=load_roster(Path(sys.argv[2]))
data=load_conditions(base/'features',roster)
for values in data.values(): validate_arrays(values)
audit=json.loads((base/'features/assembly_audit.json').read_text())
for c in data:
    assert digest(base/'features'/f'{c}.npz')==audit['files_sha256'][c], 'Feature hash mismatch'
for m in ('A','V','mask_A','mask_V','mask_T','question_presence'):
    np.testing.assert_array_equal(data['E3'][m],data['E4'][m])
print(json.dumps({c:{m:list(v[m].shape) for m in 'AVT'} for c,v in data.items()}))
print('PASS: cohort, finite values, independent zero masks, E3/E4 invariants')
""", VIEW, ZIP_DIR))

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-features/bin/python -m urop features --base /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004 --baseline /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction_excluding440/features.npz --model-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique_weights/roberta-base --zip-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --device cuda --workers 4 --stage assemble
[미실행] 설정을 바꾸면 이 명령이 실행됩니다.


{"E0": {"A": [139, 85, 88], "V": [139, 85, 272], "T": [139, 85, 768]}, "E1": {"A": [139, 85, 88], "V": [139, 85, 272], "T": [139, 85, 768]}, "E2": {"A": [139, 85, 88], "V": [139, 85, 272], "T": [139, 85, 768]}, "E3": {"A": [139, 85, 88], "V": [139, 85, 272], "T": [139, 85, 768]}, "E4": {"A": [139, 85, 88], "V": [139, 85, 272], "T": [139, 85, 768]}}
PASS: cohort, finite values, independent zero masks, E3/E4 invariants


## 8. 모델: 질문 표현과 모달리티 융합

**기초 설명**

모델은 먼저 각 모달리티 안에서 질문들의 관계를 계산하고, 다음으로 서로 다른 모달리티의 정보를 결합한다.

1. **특징 투영:** convolution으로 질문 벡터를 폭 4의 표현으로 바꾼다.
2. **위치 표현:** 질문 위치 embedding을 더해 어느 질문인지 구분한다.
3. **Self-attention:** 같은 모달리티의 다른 질문을 참조한다. 모달리티마다 두 블록을 사용한다.
4. **Cross-attention:** A–V, A–T, V–T의 세 쌍에서 서로의 정보를 참조한다.
5. **분류:** 질문 축으로 평균 pooling한 결과를 연결하고, dropout과 softmax로 두 클래스의 확률을 계산한다.

```text
A(85×88)  → Conv → 위치 → Self-attention ×2 ─┐
V(85×272) → Conv → 위치 → Self-attention ×2 ─┼→ AV / AT / VT cross-attention
T(85×768) → Conv → 위치 → Self-attention ×2 ─┘   → 평균 pooling → 연결 → dropout → softmax
```

| 논문 | GitHub `fusionmodel.py`의 구성요소 |
|---|---|
| §4.5 질문별 표현 | `FeatureEmbedding`, `PositionEmbedding`, `TransformerBlock` |
| §4.6 모달리티 융합 | `CrossTransformerBlock` |
| §4.7 분류 | `hique`의 pooling·분류층 |

**소스:** [`code/fusionmodel.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/fusionmodel.py) → `models/hique.py:build_model`.

논문 §4.3은 후속질문과 앞 질문의 topic 관계를 위치 표현에 반영한다. 공개 `PositionEmbedding`과 이 노트북의 F_AVT는 고정 위치 0–84를 사용한다.

In [10]:
# 데이터 학습 없이 모델을 구성해 구조와 수정 내역만 확인한다.
model_info = json.loads(python_read(TRAIN_PY, """
import json
from urop.models.runtime import configure_tf
configure_tf()
from urop.models.hique import build_model
from urop.models.upstream import load_patched_upstream
model=build_model('F_AVT', seed=42)
print(json.dumps({'parameters':model.count_params(),
 'inputs':[list(x.shape) for x in model.inputs], 'output':list(model.output_shape),
 'layers':[type(x).__name__ for x in model.layers],
 'compatibility_patches':load_patched_upstream().patch_manifest['patches']}))
"""))
print(json.dumps(model_info, ensure_ascii=False, indent=2))
assert model_info["parameters"] == 12554

{
  "parameters": 12554,
  "inputs": [
    [
      null,
      85,
      88
    ],
    [
      null,
      85,
      272
    ],
    [
      null,
      85,
      768
    ],
    [
      null,
      85
    ]
  ],
  "output": [
    null,
    2
  ],
  "layers": [
    "InputLayer",
    "InputLayer",
    "InputLayer",
    "FeatureEmbedding",
    "InputLayer",
    "FeatureEmbedding",
    "FeatureEmbedding",
    "SuppliedPositionEmbedding",
    "SuppliedPositionEmbedding",
    "SuppliedPositionEmbedding",
    "TransformerBlock",
    "TransformerBlock",
    "TransformerBlock",
    "TransformerBlock",
    "TransformerBlock",
    "TransformerBlock",
    "CrossTransformerBlock",
    "CrossTransformerBlock",
    "CrossTransformerBlock",
    "GlobalAveragePooling1D",
    "GlobalAveragePooling1D",
    "GlobalAveragePooling1D",
    "TFOpLambda",
    "Dropout",
    "Dense"
  ],
  "compatibility_patches": [
    "CrossTransformerBlock.get_config: return scalar constructor configuration",
    "CrossTransf

## 9. 데이터 증강과 학습

**기초 설명**

데이터 증강은 입력을 조금 바꾼 학습 예제를 추가하는 방법이다. 양성 train 표본마다 복사본 2개를 만들고, 각 복사본의 질문 10개에 해당하는 A/V/T를 함께 0으로 만든다. Dev와 test에는 증강을 적용하지 않는다.

| 설정 | 값 | 의미 |
|---|---|---|
| Epochs | 100 | 학습 데이터를 반복해서 보는 횟수 |
| Batch size | 8 | 한 번의 가중치 업데이트에 사용하는 예제 수 |
| Optimizer / learning rate | Adam / 0.0002 | 가중치 업데이트 방법과 크기 |
| Dropout | 0.5 | 학습 중 일부 표현을 생략해 과도한 의존을 줄이는 비율 |
| 모델 선택 | 최저 dev loss | dev에서 분류 손실이 가장 작은 가중치 저장 |

같은 seed는 초기화·증강의 난수 조건을 정한다. 다섯 입력 조건에 다섯 seed를 적용해 총 25회 학습한다. 아래 두 셀은 처음 15회와 추가 10회로 나뉜다.

**논문·코드 연결**

- 논문: §3.1: 양성 표본 증강; §5: 학습 설정
- GitHub: [`code/hique.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/hique.py)의 학습·checkpoint 저장 흐름
- 실행 구현: `training/augmentation.py:augment`, `training/runner.py:train`

증강은 논문 §3.1을 기준으로 구현한다. Epochs는 논문의 100회를 사용하며 공개 CLI 기본값은 50회다.

In [11]:
provenance = [BASE / name for name in (
    "protocol.json", "notebook_recipe.json", "mapping.json", "mapping_review.json",
    "train_utterance_inventory.json", "manifest.json", "preparation_summary.json",
    "input_review_packet.json", "events.jsonl", "segments_E0.jsonl", "segments_E1.jsonl",
    "segments_E2.jsonl", "segments_E3.jsonl", "features/frozen_inputs.json",
    "features/text_audit.json", "features/assembly_audit.json")]
provenance += [QUESTIONS]
train_args = ["--features-dir", BASE / "features", "--label-dir", ZIP_DIR,
              "--output-dir", BASE / "runs"]
for path in provenance:
    train_args += ["--provenance-file", path]
run_cli(TRAIN_PY, "train", *train_args, "--stage", "initial")

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-tf/bin/python -m urop train --features-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/features --label-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --output-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/runs --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/protocol.json --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/notebook_recipe.json --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/mapping.json --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/mapping_review.json --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/train_utterance_inventory.json --provenance-file /home/ubuntu/Des

In [12]:
run_cli(TRAIN_PY, "train", *train_args, "--stage", "extend")

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-tf/bin/python -m urop train --features-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/features --label-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --output-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/runs --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/protocol.json --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/notebook_recipe.json --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/mapping.json --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/mapping_review.json --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/train_utterance_inventory.json --provenance-file /home/ubuntu/Des

## 10. Dev 결과와 대조군

**기초 설명**

다섯 seed의 점수를 평균하면 여러 초기화에서의 성능을 볼 수 있다. 함께 표시하는 SD는 **seed 사이의 표준편차**로, 값이 클수록 실행에 따라 성능이 많이 달라진다. Dev는 모델 선택에도 사용하므로 이 결과는 탐색적으로 해석한다.

대조군은 복잡한 모델의 점수를 비교할 기준이다.

| 대조군 | 입력·예측 방식 | 확인할 점 |
|---|---|---|
| 질문 존재 대조군 | 질문 유무 85개로 logistic regression 학습 | 답변 내용 없이 질문 패턴만으로 예측되는가? |
| 다수 클래스 대조군 | train에서 더 많은 클래스를 항상 예측 | 단순한 예측 기준보다 나은가? |

**논문·코드 연결**

- 논문: §5.2·Table 1: 분류 지표로 성능 비교
- GitHub: [`code/hique.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/hique.py)의 분류 평가
- 실행 구현: `training/runner.py:aggregate/write_controls`, `training/report.py:render`

위 두 대조군은 이 노트북의 입력 비교를 위한 보조 기준이다.

In [13]:
run_cli(TRAIN_PY, "report", "--base", BASE)
dev = read_json(VIEW / "runs/extend_results.json")
completion = read_json(VIEW / "runs/extend_complete.json")
assert digest(VIEW / "runs/extend_results.json") == completion["results_sha256"]
rows = []
for c in CONDITIONS:
    summary = dev["conditions"][c]["summary"]["dev"]
    f = summary["macro_f1"]
    rows.append([c, f"{f['mean']:.4f} ± {f['std']:.4f}",
                 f"{summary['sensitivity']['mean']:.4f}", f"{summary['specificity']['mean']:.4f}"])
display(Markdown("**새 실행 dev 결과**" if RUN_PIPELINE else "**기존 V3 dev 결과 — 이번 노트북에서 재학습한 값이 아님**"))
table(["조건", "Macro-F1 평균 ± SD", "민감도", "특이도"], rows)
table(["비교", "Macro-F1 차이", "참가자 재표집 95% 구간"],
      [[k, f"{v['delta_macro_f1']:+.4f}", str(v["ci95"])] for k,v in dev["comparisons"].items()])
table(["조건", "질문 존재 대조군 Macro-F1", "다수 클래스 Macro-F1"],
      [[c, f"{dev['controls'][c]['question_presence']['macro_f1']:.4f}",
        f"{dev['controls'][c]['majority']['macro_f1']:.4f}"] for c in CONDITIONS])

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-tf/bin/python -m urop report --base /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004
[미실행] 설정을 바꾸면 이 명령이 실행됩니다.


**기존 V3 dev 결과 — 이번 노트북에서 재학습한 값이 아님**

| 조건 | Macro-F1 평균 ± SD | 민감도 | 특이도 |
|---|---|---|---|
| E0 | 0.5656 ± 0.0369 | 0.3636 | 0.7714 |
| E1 | 0.6730 ± 0.0786 | 0.5091 | 0.8286 |
| E2 | 0.6838 ± 0.0678 | 0.5091 | 0.8571 |
| E3 | 0.6462 ± 0.0703 | 0.4364 | 0.8476 |
| E4 | 0.6594 ± 0.0520 | 0.4909 | 0.8286 |

| 비교 | Macro-F1 차이 | 참가자 재표집 95% 구간 |
|---|---|---|
| E1_minus_E0 | +0.1074 | [-0.0562416476510833, 0.2774403813571829] |
| E2_minus_E1 | +0.0108 | [-0.05644942409142377, 0.08405292693971364] |
| E3_minus_E1 | -0.0268 | [-0.07975358430155584, 0.028159427964382568] |
| E4_minus_E3 | +0.0132 | [-0.036590295890053, 0.0657375867535753] |

| 조건 | 질문 존재 대조군 Macro-F1 | 다수 클래스 Macro-F1 |
|---|---|---|
| E0 | 0.6632 | 0.3962 |
| E1 | 0.6082 | 0.3962 |
| E2 | 0.6082 | 0.3962 |
| E3 | 0.6082 | 0.3962 |
| E4 | 0.6082 | 0.3962 |

## 11. Test 평가 설정

**기초 설명**

Test는 학습에 사용하지 않은 참가자에게 모델을 적용하는 단계다. 평가 전에 입력 처리, 가중치, 분류 기준을 정한다. 여기서는 dev 평균 Macro-F1이 가장 높은 조건을 참고 조건으로 선택하고, 다섯 조건의 고정 모델을 모두 평가한다.

`RUN_PIPELINE=True`, `RUN_TEST=True`일 때 아래 셀이 실행된다. 양성 판정 기준은 `p1 > 0.5`다. 이 데이터에는 기존 test 결과 노출 이력이 있으므로 결과는 참고 benchmark로 해석한다.

**논문·코드 연결**

- 논문: §3·§5: 분리된 test 집합의 평가
- GitHub: [`code/hique.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/hique.py)의 모델 선택·test 평가 흐름
- 실행 구현: `evaluation/runner.py:verify_training/snapshot/freeze`

In [14]:
TEST_BASE = BASE / "test_evaluation"
TEST_ENABLED = RUN_PIPELINE and RUN_TEST
if TEST_ENABLED:
    print(python_read(TRAIN_PY, """
from pathlib import Path
import sys
from urop.evaluation.runner import verify_training
verify_training(Path(sys.argv[1]))
print('PASS: 25 complete unchanged training runs')
""", BASE / "runs"))
    with (ZIP_DIR / "test_split_Depression_AVEC2017.csv").open(newline="") as stream:
        reader = csv.DictReader(stream)
        column = next(k for k in reader.fieldnames if k.strip().lower() == "participant_id")
        official_test_ids = [int(r[column]) for r in reader if r.get(column, "").strip()]
    assert len(official_test_ids) == len(set(official_test_ids)) == 47
    assert {411, 480}.issubset(official_test_ids)
    test_ids = sorted(set(official_test_ids) - {411, 480})
    assert len(test_ids) == 45 and 300 in test_ids
    reference = sorted(CONDITIONS, key=lambda c: (-dev["conditions"][c]["summary"]["dev"]["macro_f1"]["mean"], c))[0]
    write_once(TEST_BASE / "test_protocol.json", {
        "cohort": {"test_ids": test_ids, "n": 45, "excluded": [411, 480]},
        "conditions": CONDITIONS, "seeds": SEEDS, "prior_test_exposure": True,
        "training_config_sha256": digest(BASE / "runs/frozen_config.json"),
        "training_complete_sha256": digest(BASE / "runs/extend_complete.json"),
        "pretest_dev_selected_reference": reference,
        "claim_limit": "Reference benchmark with prior test exposure; not exact paper reproduction",
        "selection_rule": "highest mean dev macro_f1; lexicographic tie break",
        "threshold": "p1 > 0.5; no test tuning"})
    print("Test 전에 고정한 dev 참고 조건:", reference)
else:
    print("새 test 프로토콜 생성 안 함. 뒤의 test 실행 명령도 미실행 상태입니다.")

새 test 프로토콜 생성 안 함. 뒤의 test 실행 명령도 미실행 상태입니다.


## 12. Test 특징 구성

**기초 설명**

Test 입력에도 학습 때와 같은 질문 사전, 답변 선택 규칙, 특징 추출기를 적용한다. 같은 종류의 입력을 같은 숫자 표현으로 바꿔야 학습된 가중치를 사용할 수 있다.

아래 네 셀은 **질문·답변 준비 → 음성·얼굴 특징 → 텍스트 특징 → 배열 조립** 순서다. 결과는 `test_evaluation/features/`에 저장한다.

**논문·코드 연결**

- 논문: §4.3–4.4: 질문 구성과 특징 추출
- GitHub: [`code/preprocess.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/preprocess.py), [`code/feature_extraction/extract_audio.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/feature_extraction/extract_audio.py), [`code/feature_extraction/extract_visual.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/feature_extraction/extract_visual.py), [`embedding/roberta.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/embedding/roberta.py)
- 실행 구현: `evaluation/test_features.py:prepare/extract_av/assemble`, `features/text.py:extract_text`

In [15]:
test_feature_args = ["--base", TEST_BASE, "--training", BASE, "--baseline", BASELINE,
                     "--baseline-segments", BASELINE_SEGMENTS, "--questions", QUESTIONS,
                     "--model-dir", MODEL_DIR, "--zip-dir", ZIP_DIR,
                     "--device", DEVICE, "--workers", AV_WORKERS]
run_cli(FEATURE_PY, "test-features", *test_feature_args, "--stage", "prepare", enabled=TEST_ENABLED)

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-features/bin/python -m urop test-features --base /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation --training /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004 --baseline /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction_excluding440/features.npz --baseline-segments /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction_excluding440/segments.jsonl --questions /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction/questions.json --model-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique_weights/roberta-base --zip-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --device cuda --workers 4 --stage prepare
[미실행] 설정을 바꾸면 이 명령이 실행됩니다.


In [16]:
run_cli(FEATURE_PY, "test-features", *test_feature_args, "--stage", "av", enabled=TEST_ENABLED)

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-features/bin/python -m urop test-features --base /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation --training /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004 --baseline /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction_excluding440/features.npz --baseline-segments /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction_excluding440/segments.jsonl --questions /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction/questions.json --model-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique_weights/roberta-base --zip-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --device cuda --workers 4 --stage av
[미실행] 설정을 바꾸면 이 명령이 실행됩니다.


In [17]:
run_cli(FEATURE_PY, "test-features", *test_feature_args, "--stage", "text", enabled=TEST_ENABLED)

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-features/bin/python -m urop test-features --base /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation --training /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004 --baseline /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction_excluding440/features.npz --baseline-segments /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction_excluding440/segments.jsonl --questions /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction/questions.json --model-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique_weights/roberta-base --zip-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --device cuda --workers 4 --stage text
[미실행] 설정을 바꾸면 이 명령이 실행됩니다.


In [18]:
run_cli(FEATURE_PY, "test-features", *test_feature_args, "--stage", "assemble", enabled=TEST_ENABLED)

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-features/bin/python -m urop test-features --base /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation --training /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004 --baseline /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction_excluding440/features.npz --baseline-segments /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction_excluding440/segments.jsonl --questions /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/hique_reproduction/questions.json --model-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique_weights/roberta-base --zip-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --device cuda --workers 4 --stage assemble
[미실행] 설정을 바꾸면 이 명령이 실행됩니다.


## 13. 저장된 모델로 Test 예측

**기초 설명**

Checkpoint는 학습 중 저장한 가중치다. 모델 구조를 구성하고 최저 dev loss의 가중치를 불러온 뒤, test 참가자별 클래스 확률을 계산한다. 이 단계에서는 가중치를 학습하지 않는다.

첫 셀의 `freeze`는 평가에 사용할 설정과 파일을 고정하고, 둘째 셀의 `evaluate`는 예측과 지표 계산을 수행한다. 완료한 결과를 다시 볼 때는 다음 집계 셀을 사용한다.

**논문·코드 연결**

- 논문: §5.2: 분류 성능 평가
- GitHub: [`code/hique.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/hique.py)의 저장 모델 로딩·test 예측
- 실행 구현: `evaluation/runner.py:freeze/evaluate`

In [19]:
eval_args = ["--training-dir", BASE / "runs", "--features-dir", TEST_BASE / "features",
             "--label-dir", ZIP_DIR, "--output-dir", TEST_BASE / "runs",
             "--protocol", TEST_BASE / "test_protocol.json",
             "--tokenizer-provenance", TEST_BASE / "features/text_audit.json",
             "--feature-code", ROOT / "src/urop/evaluation/test_features.py"]
for filename in ("manifest.json", "preparation_summary.json", "segments_E0.jsonl", "segments_E1.jsonl",
                 "segments_E2.jsonl", "segments_E3.jsonl", "features/frozen_inputs.json",
                 "features/assembly_audit.json"):
    eval_args += ["--provenance-file", TEST_BASE / filename]
run_cli(TRAIN_PY, "evaluate", *eval_args, "--stage", "freeze", enabled=TEST_ENABLED)

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-tf/bin/python -m urop evaluate --training-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/runs --features-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation/features --label-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --output-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation/runs --protocol /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation/test_protocol.json --tokenizer-provenance /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation/features/text_audit.json --feature-code /home/ubuntu/Desktop/UROP_Depressive_Dectection/src/urop/evaluation/test_features.py --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation/manife

In [20]:
run_cli(TRAIN_PY, "evaluate", *eval_args, "--stage", "evaluate", enabled=TEST_ENABLED)

/home/ubuntu/Desktop/UROP_Depressive_Dectection/.tmp/hique-tf/bin/python -m urop evaluate --training-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/runs --features-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation/features --label-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/DAIC-WOZ --output-dir /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation/runs --protocol /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation/test_protocol.json --tokenizer-provenance /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation/features/text_audit.json --feature-code /home/ubuntu/Desktop/UROP_Depressive_Dectection/src/urop/evaluation/test_features.py --provenance-file /home/ubuntu/Desktop/UROP_Depressive_Dectection/Data/runs/hique_notebook_20261004/test_evaluation/manife

## 14. 평가 지표 읽기

**기초 설명**

정답과 예측을 비교하면 네 경우가 생긴다. **TP**는 양성을 양성으로, **TN**은 음성을 음성으로 맞힌 수다. **FP**는 음성을 양성으로, **FN**은 양성을 음성으로 잘못 예측한 수다. 여기서 양성·음성은 PHQ-8 선별 라벨을 뜻한다.

| 지표 | 계산·의미 |
|---|---|
| Precision | `TP / (TP + FP)` — 양성 예측 중 실제 양성 비율 |
| Recall / 민감도 | `TP / (TP + FN)` — 실제 양성 중 찾아낸 비율 |
| 특이도 | `TN / (TN + FP)` — 실제 음성 중 맞힌 비율 |
| 양성 F1 | `2TP / (2TP + FP + FN)` — precision과 recall을 함께 요약 |
| **Macro-F1** | 두 클래스 F1의 단순 평균 — 각 클래스에 같은 비중 부여 |
| Weighted-F1 | 클래스별 표본 수에 비례해 F1을 평균 |
| AUROC | 여러 분류 기준에서 양성·음성을 구분하는 순위 능력 |

조건 간 차이에는 참가자를 2,000회 재표집하는 **paired bootstrap**을 사용한다. 비교 조건에서 같은 사람을 함께 뽑아 Macro-F1 차이의 95% 구간을 계산한다. 구간에 0이 포함되면 어느 조건이 우세한지 명확하지 않다. 이 구간과 seed 간 SD는 서로 다른 변동을 나타낸다.

**논문·코드 연결**

- 논문: §5.2·Table 1: precision·recall·macro/weighted F1 등의 비교
- GitHub: [`code/hique.py`](https://github.com/JuHo-Jung/HiQuE/blob/24c553bf2666b442ae5b0e3490b998a5d4493559/code/hique.py)의 분류 지표 계산
- 실행 구현: `evaluation/metrics.py:metrics/paired_bootstrap`

민감도·특이도·AUROC와 bootstrap 구간은 이 노트북에서 함께 보는 보조 지표다. 아래 셀은 가상 예제의 지표와 실험 집계 결과를 표시한다.

In [21]:
# 가상 4명으로 지표 계산을 확인한다. 실제 test 라벨이 아니다.
print(python_read(TRAIN_PY, """
import json
from urop.evaluation.metrics import metrics
print(json.dumps(metrics([0,0,1,1], [0.1,0.2,0.8,0.4])))
"""))
if TEST_ENABLED:
    run_cli(TRAIN_PY, "test-report", "--base", BASE, enabled=True)
# 읽기 모드에서만 과거 test 결과를 표시한다. 새 train-only 실행에 과거 결과를 섞지 않는다.
test_view = VIEW / "test_evaluation"
if not RUN_PIPELINE or TEST_ENABLED:
    test = read_json(test_view / "runs/test_results.json")
    manifest = read_json(test_view / "runs/evaluation_manifest.json")
    assert manifest["status"] == "complete"
    assert digest(test_view / "runs/test_results.json") == manifest["test_results_sha256"]
    display(Markdown("**새 실행의 test 결과**" if TEST_ENABLED else "**기존 V3 test 결과 — 재평가하지 않고 읽은 값**"))
    rows=[]
    for c in CONDITIONS:
        s=test["conditions"][c]["per_seed_summary"]
        rows.append([c, *[f"{s[k]['mean']:.4f} ± {s[k]['std']:.4f}" for k in
                          ("macro_f1", "positive_f1", "sensitivity", "specificity", "auroc")]])
    table(["조건", "Macro-F1", "양성 F1", "민감도", "특이도", "AUROC"], rows)
    print("Test 이전 dev 참고 조건:", test["pretest_dev_selected_reference"])
    table(["비교", "Macro-F1 차이", "참가자 재표집 95% 구간"],
          [[k,f"{v['delta_macro_f1']:+.4f}",str(v["ci95"])] for k,v in test["comparisons"].items()])
else:
    print("이번 실행은 train/dev까지만 수행했습니다. 새로운 test 결과는 없습니다.")

{"n": 4, "macro_f1": 0.7333333333333334, "positive_f1": 0.6666666666666666, "sensitivity": 0.5, "specificity": 1.0, "auroc": 1.0, "confusion_matrix": [[2, 0], [1, 1]], "macro_precision": 0.8333333333333333, "macro_recall": 0.75, "weighted_precision": 0.8333333333333333, "weighted_recall": 0.75, "weighted_f1": 0.7333333333333334, "gmean": 0.7071067811865476}


**기존 V3 test 결과 — 재평가하지 않고 읽은 값**

| 조건 | Macro-F1 | 양성 F1 | 민감도 | 특이도 | AUROC |
|---|---|---|---|---|---|
| E0 | 0.5920 ± 0.0585 | 0.4339 ± 0.1225 | 0.4571 ± 0.1948 | 0.7548 ± 0.1300 | 0.6433 ± 0.0869 |
| E1 | 0.6490 ± 0.0685 | 0.5310 ± 0.0810 | 0.5571 ± 0.1143 | 0.7548 ± 0.1268 | 0.7207 ± 0.0791 |
| E2 | 0.6541 ± 0.0344 | 0.5471 ± 0.0596 | 0.6143 ± 0.1161 | 0.7226 ± 0.0562 | 0.7392 ± 0.0674 |
| E3 | 0.6365 ± 0.0453 | 0.5088 ± 0.0569 | 0.5286 ± 0.0969 | 0.7548 ± 0.0970 | 0.7069 ± 0.0362 |
| E4 | 0.6431 ± 0.0786 | 0.5313 ± 0.0838 | 0.5714 ± 0.0782 | 0.7290 ± 0.1052 | 0.6899 ± 0.0626 |

Test 이전 dev 참고 조건: E2, highestV3mean devMacroF1; no posttest modelselection


| 비교 | Macro-F1 차이 | 참가자 재표집 95% 구간 |
|---|---|---|
| E1_minus_E0 | +0.0570 | [-0.06420620927734796, 0.18167326525809516] |
| E2_minus_E1 | +0.0051 | [-0.060318971985867366, 0.07133166746426875] |
| E3_minus_E1 | -0.0125 | [-0.07979038711609554, 0.05507422042377539] |
| E4_minus_E3 | +0.0066 | [-0.04383243846666367, 0.05802437581013101] |

## 15. 결과 파일 안내

모든 새 실행 결과는 `Data/runs/<RUN_NAME>/` 아래에 저장된다.

| 경로 | 내용 |
|---|---|
| `segments_E*.jsonl` | 질문별 답변과 시간 구간 |
| `features/E*.npz` | 모델에 입력하는 특징 배열 |
| `runs/<condition>_seed<seed>/` | 모델 가중치·학습 곡선·예측 |
| `results.csv`, `results_ko.md` | Dev 집계 결과 |
| `test_evaluation/` | Test 특징·예측·집계 결과 |

[실행 방법](../../../docs/reproducibility.md) · [실험 조건 상세](README.md) · [논문·구현 상세 비교](../../../docs/hique_diagnosis.md)